<a href="https://colab.research.google.com/github/adikatre/Asymmetric-Cross-Modal-Attention/blob/main/notebooks/02_train_generative_frozen_asymmetric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load and Unzip VQA Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/sample_data/

Mounted at /content/drive


In [ ]:
# Get vqa data from drive opt.
!cp "/content/drive/MyDrive/VQA.zip" "/content/"

In [2]:
# make data dirs
!mkdir -p /content/data/
!mkdir -p /content/data/answers
!mkdir -p /content/data/images
!mkdir -p /content/data/questions

#copy over zips (https://drive.google.com/drive/folders/1VJ1xNxo_dAGJ4ZcpaFQBo-wpdIChZkpx?usp=sharing) from drive into here
!cp -r /content/drive/MyDrive/VQA/ /content/data/zip/

In [ ]:
# unqip vqa.zip, then remove it opt.
!unzip -q /content/VQA.zip -d /content/data/zip

!rm -rf /content/VQA.zip

In [ ]:
# test2015 opt.
!unzip -q /content/data/zip/test2015.zip -d /content/data/images/

!rm -rf /content/data/zip/test2015.zip

In [ ]:
# train2014 opt.
!unzip -q /content/data/zip/train2014.zip -d /content/data/images/

!rm -rf /content/data/zip/train2014.zip

In [ ]:
# val2014 opt.
!unzip -q /content/data/zip/val2014.zip -d /content/data/images/

!rm -rf /content/data/zip/val2014.zip

In [3]:
# extract annotations (labels)
!unzip -q /content/data/zip/v2_Annotations_Train_mscoco.zip -d /content/data/answers/
!unzip -q /content/data/zip/v2_Annotations_Val_mscoco.zip -d /content/data/answers/

!rm -rf /content/data/zip/v2_Annotations_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Annotations_Val_mscoco.zip

In [4]:
# extract testing data
!unzip -q /content/data/zip/v2_Questions_Test_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Train_mscoco.zip -d /content/data/questions/
!unzip -q /content/data/zip/v2_Questions_Val_mscoco.zip -d /content/data/questions/

!rm -rf /content/data/zip/v2_Questions_Test_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Train_mscoco.zip
!rm -rf /content/data/zip/v2_Questions_Val_mscoco.zip

In [ ]:
# copy over past results for resume function
!mkdir -p /content/results
!cp -r /content/drive/MyDrive/unfrozen_results_4_25/final_results/* /content/results/

In [5]:
# copy over vqa images h5 from VQA dir
!cp /content/drive/MyDrive/VQA_cache/vqa_images_336.h5 /content/data/


# 2 — Train, Evaluate & Visualize

Complete experiment in one notebook:

1. **Data** — Load VQA v2.0 dataset
2. **Models** — Define encoders, attention blocks, and full VQA models
3. **Training** — Train symmetric baseline and asymmetric model
4. **Evaluation** — Compare metrics (Top-1, Top-5 accuracy)
5. **Visualization** — Training curves, attention heatmaps, qualitative examples

**Run all cells in order.** Edit the configuration cell below to change hyperparameters.

In [6]:
!pip install -q torch torchvision transformers sentencepiece timm matplotlib tqdm Pillow h5py open_clip_torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00


In [7]:
import json
import random
import time
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

import open_clip
from transformers import T5ForConditionalGeneration, T5TokenizerFast, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print(f"Device: {device}")


Device: cuda


## Configuration

Edit these variables to change the experiment.

In [8]:
# Paths
DATA_DIR       = Path("/content/data")
CHECKPOINT_DIR = Path("/content/results/checkpoints")
METRICS_DIR    = Path("/content/results/metrics")
FIGURES_DIR    = Path("/content/results/figures")

# Fusion variant. THIS IS THE ONLY LINE THAT DIFFERS between the asymmetric
# and symmetric sibling notebooks. Everything downstream branches off it.
FUSION_TYPE = "asymmetric"   # "asymmetric" or "symmetric"

# Model
EMBED_DIM        = 1024      # FlanT5-large hidden size; fusion + decoder share this
NUM_HEADS        = 16
FUSION_DEPTH     = 6         # number of (self-attn + cross-attn) fusion layers
DROPOUT          = 0.1
ATTN_DROPOUT     = 0.1
CLS_DROPOUT      = 0.3       # unused with generative head; kept for arg-compat
VISION_MODEL     = "ViT-bigG-14"
VISION_PRETRAINED = "laion2b_s39b_b160k"
VISION_DIM       = 1664       # ViT-bigG-14 transformer width; tokens from output_tokens=True are NOT projected to embed_dim (1280)
VISION_TOKENS    = 256        # 16*16 patches; open_clip._pool extracts CLS into `pooled`, so tokens are patches only
T5_MODEL         = "google/flan-t5-large"
ANSWER_MAX_LEN   = 10         # max decoded answer tokens

# Data
MAX_QUESTION_LEN = 20
MAX_SAMPLES      = None       # set to a small int for a dev run
QUESTION_PROMPT  = "Answer the question using a single word or short phrase: {question}"

# Training (encoders frozen + vision features precomputed -> A100 can run larger batch)
BATCH_SIZE       = 128        # ViT-bigG-14 forward removed from train loop on A100; staying at 128 to match prior LR regime
LEARNING_RATE    = 1e-4       # fusion LR (matches staged BS=128 regime)
PROJ_LR          = 1e-3       # vision/text projection LR (matches staged BS=128 regime)
WEIGHT_DECAY     = 1e-2
EPOCHS           = 18
WARMUP_EPOCHS    = 1
EARLY_STOP_PATIENCE = 3       # stop if val subset acc doesn't improve for N epochs
EVAL_SUBSET_SIZE = 10000      # per-epoch fast val subset
NUM_WORKERS      = 4          # dropped from 8: per-step CPU work is tiny once vision feats are cached
PREFETCH_FACTOR  = 2          # dropped from 4 for the same reason
SEED             = 42
USE_AMP          = True
AMP_DTYPE        = torch.bfloat16

for d in [CHECKPOINT_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"FUSION_TYPE={FUSION_TYPE} | BATCH_SIZE={BATCH_SIZE} | EPOCHS={EPOCHS}")


FUSION_TYPE=asymmetric | BATCH_SIZE=128 | EPOCHS=18


---
## 1. Data Loading

In [9]:
import re

# VQAv2 official-style answer normalization (lowercase, strip punctuation,
# remove articles, fix common contractions). Applied identically when building
# the vocabulary and when constructing soft targets so they stay in sync.
_ARTICLES = {"a", "an", "the"}
_PUNCT_RE = re.compile(r"[^\w\s\']")
_CONTRACTIONS = {
    "dont": "don't", "doesnt": "doesn't", "didnt": "didn't",
    "isnt": "isn't", "arent": "aren't",
    "wasnt": "wasn't", "werent": "weren't",
    "wont": "won't", "cant": "can't", "couldnt": "couldn't",
    "wouldnt": "wouldn't", "shouldnt": "shouldn't",
    "havent": "haven't", "hasnt": "hasn't", "hadnt": "hadn't",
    "thats": "that's", "whats": "what's", "wheres": "where's",
    "theres": "there's", "heres": "here's", "youre": "you're",
    "theyre": "they're", "weve": "we've", "youve": "you've",
}


def normalize_answer(s: str) -> str:
    """Lowercase, strip punctuation, drop articles, fix contractions."""
    s = s.lower().strip()
    s = _PUNCT_RE.sub(" ", s)
    s = " ".join(t for t in s.split() if t not in _ARTICLES)
    return _CONTRACTIONS.get(s, s)


# OpenCLIP ViT-bigG-14 was trained at 224x224 with its own normalization stats.
# We use the official preprocess returned by open_clip to stay faithful to the
# pretrained distribution. Train aug = light random crop + horizontal flip is
# deliberately omitted (flip inverts left/right which matters for VQA).
OPENAI_CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
OPENAI_CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)


def get_image_transform(split="val"):
    if split == "train":
        return transforms.Compose([
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=OPENAI_CLIP_MEAN, std=OPENAI_CLIP_STD),
        ])
    return transforms.Compose([
        transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=OPENAI_CLIP_MEAN, std=OPENAI_CLIP_STD),
    ])


def get_h5_preprocess_transform():
    """Resize to 336x336 for HDF5 storage (kept compatible with existing cache)."""
    return transforms.Compose([
        transforms.Resize(336),
        transforms.CenterCrop(336),
    ])


## Storage as one .h5 file

resized to 256x256 before adding to .h5 file

In [10]:
# --- HDF5 Preprocessing: convert raw JPEGs into a single contiguous file ---
h5_path = DATA_DIR / "vqa_images_336.h5"

if not h5_path.exists():
    preprocess = get_h5_preprocess_transform()
    image_dirs = [DATA_DIR / "images" / "train2014", DATA_DIR / "images" / "val2014"]

    # Collect all image paths
    all_paths = []
    for d in image_dirs:
        if d.exists():
            all_paths.extend(sorted(d.glob("*.jpg")))
    print(f"Found {len(all_paths):,} images to preprocess")

    with h5py.File(h5_path, "w") as h5f:
        imgs_ds = h5f.create_dataset(
            "images", shape=(len(all_paths), 336, 336, 3),
            dtype=np.uint8, chunks=(1, 336, 336, 3))
        ids_ds = h5f.create_dataset(
            "image_ids", shape=(len(all_paths),), dtype=np.int64)

        for i, path in enumerate(tqdm(all_paths, desc="Preprocessing images")):
            image_id = int(path.stem.split("_")[-1])
            img = Image.open(path).convert("RGB")
            img = preprocess(img)
            imgs_ds[i] = np.array(img)
            ids_ds[i] = image_id

    print(f"Saved {len(all_paths):,} images to {h5_path}")
else:
    print(f"HDF5 file already exists: {h5_path}")


HDF5 file already exists: /content/data/vqa_images_336.h5


In [11]:
class VQADataset(Dataset):
    """VQA v2.0 dataset for generative training. Consumes PRECOMPUTED vision
    features (ViT-bigG-14 outputs) cached in an HDF5 file.

    Returns (vision_feats, question_input_ids, question_attention_mask,
            label_input_ids, answers_list_json).

    - vision_feats: (Nv, 1664) float tensor read from vqa_vision_feats_bigG.h5.
    - question_input_ids / mask: FlanT5 tokenization of the prompt-wrapped question.
    - label_input_ids: FlanT5 tokenization of the gold target answer (most-agreed
      annotator answer), with pad positions set to -100 for the LM loss.
    - answers_list_json: JSON-encoded list of the 10 normalized annotator answers
      (for the official VQA soft-accuracy at eval time).

    Tokenization and image features are both deterministic for a given sample,
    so they are precomputed once -- the per-step pipeline is now just h5 reads
    plus a copy to GPU.
    """

    def __init__(self, questions_file, annotations_file, vision_feats_h5_path,
                 t5_tokenizer, max_question_len=20, answer_max_len=10,
                 max_samples=None):
        self.feats_h5_path = Path(vision_feats_h5_path)
        self.max_question_len = max_question_len
        self.answer_max_len = answer_max_len
        self.tokenizer = t5_tokenizer
        self._feats_h5 = None

        with h5py.File(self.feats_h5_path, "r") as f:
            image_ids = f["image_ids"][:]
        self.id_to_row = {int(iid): i for i, iid in enumerate(image_ids)}

        with open(questions_file) as f:
            questions_data = json.load(f)["questions"]
        with open(annotations_file) as f:
            annotations_data = json.load(f)["annotations"]
        ann_by_qid = {ann["question_id"]: ann for ann in annotations_data}

        self.samples = []
        for q in questions_data:
            ann = ann_by_qid.get(q["question_id"])
            if ann is None:
                continue
            norm_answers = [normalize_answer(a["answer"]) for a in ann["answers"]]
            # Training target: most-agreed answer (ties broken by first occurrence).
            target = Counter(norm_answers).most_common(1)[0][0]
            self.samples.append({
                "question": q["question"],
                "image_id": q["image_id"],
                "target_answer": target,
                "all_answers": norm_answers,
            })
            if max_samples is not None and len(self.samples) >= max_samples:
                break

        # Text is deterministic for a fixed sample, so cache tokenization once.
        prompts = [QUESTION_PROMPT.format(question=s["question"]) for s in self.samples]
        targets = [s["target_answer"] for s in self.samples]
        if self.samples:
            q_enc = self.tokenizer(
                prompts, max_length=self.max_question_len, padding="max_length",
                truncation=True, return_tensors="pt",
            )
            a_enc = self.tokenizer(
                targets, max_length=self.answer_max_len,
                padding="max_length", truncation=True, return_tensors="pt",
            )
            label_ids = a_enc["input_ids"].masked_fill(
                a_enc["input_ids"] == self.tokenizer.pad_token_id, -100)
            for i, sample in enumerate(self.samples):
                sample["q_input_ids"] = q_enc["input_ids"][i]
                sample["q_attention_mask"] = q_enc["attention_mask"][i]
                sample["label_ids"] = label_ids[i]
                sample["answers_json"] = json.dumps(sample["all_answers"])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        if self._feats_h5 is None:
            # Open lazily so the file handle is created in each DataLoader worker.
            self._feats_h5 = h5py.File(self.feats_h5_path, "r")
        row = self.id_to_row[sample["image_id"]]
        feats = self._feats_h5["features"][row]  # (Nv, 1664) fp16
        vision_feats = torch.from_numpy(feats).float()

        return (
            vision_feats,
            sample["q_input_ids"].clone(),
            sample["q_attention_mask"].clone(),
            sample["label_ids"].clone(),
            sample["answers_json"],
        )


In [12]:
# Tokenizer is built here; DataLoaders are built AFTER the vision-feature
# precomputation cell below, since they need the precomputed-features h5.
t5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

---
## 2. Model Definitions

### Encoders

Both encoders are **frozen** (pretrained weights fixed). Only the fusion layers and classifier train.

- **ImageEncoder** — DINOv2 ViT-g/14: produces 577 tokens (576 spatial + 1 CLS) at 336x336, projected from 1536 → `EMBED_DIM`
- **TextEncoder** — DeBERTa-v3-large: produces per-token embeddings, projected from 1024 → `EMBED_DIM`

In [13]:
class FrozenVisionEncoder(nn.Module):
    """Frozen OpenCLIP ViT-bigG-14 vision tower returning patch tokens (B, 257, 1280)."""

    def __init__(self, model_name=VISION_MODEL, pretrained=VISION_PRETRAINED):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            model_name, pretrained=pretrained
        )
        self.visual = clip_model.visual
        # Ask open_clip to return token-level features alongside the pooled embedding.
        # On VisionTransformer.forward: when output_tokens=True it returns (pooled, tokens).
        self.visual.output_tokens = True
        for p in self.visual.parameters():
            p.requires_grad = False
        self.visual.eval()

    @torch.inference_mode()
    def forward(self, images):
        # Returns (pooled, tokens). We want the token sequence including the CLS-like
        # learned query at position 0 for cross-modal attention.
        _pooled, tokens = self.visual(images)
        return tokens  # (B, 256 or 257, 1280)


class FrozenTextEncoder(nn.Module):
    """Wrapper around a frozen T5 encoder for question tokens. Output: (B, L, 1024)."""

    def __init__(self, t5_model):
        super().__init__()
        self.encoder = t5_model.encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.encoder.eval()

    @torch.inference_mode()
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state  # (B, L, 1024)


### Cross-Attention Blocks

The core research contribution. Cross-attention lets one modality "ask questions" of the other:

- **Asymmetric**: Two **independent** blocks with separate weights — one for image→text, one for text→image
- **Symmetric** (baseline): A **single shared** block used in both directions — cannot learn directional patterns

In [14]:
class CrossAttentionBlock(nn.Module):
    """Cross-attention: queries from one modality attend to keys/values from another.
    Includes LayerNorm, residual connections, and a feed-forward network.

    Two dropout knobs:
      - attn_dropout: dropout INSIDE the multi-head attention softmax
      - dropout: dropout in the FFN sub-block
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None):
        super().__init__()
        if attn_dropout is None:
            attn_dropout = dropout
        # Pre-norm design: LayerNorm is applied BEFORE attention (not after).
        # This improves training stability for deep networks compared to post-norm.
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_kv = nn.LayerNorm(embed_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=attn_dropout, batch_first=True)
        self.norm_ff = nn.LayerNorm(embed_dim)
        # Standard Transformer FFN: expand 4x with GELU activation, then project back
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, query, key_value, key_padding_mask=None):
        # Pre-norm cross-attention: normalize inputs, attend, then ADD back the
        # original query (residual). This preserves the raw query signal while
        # enriching it with cross-modal context from key_value.
        q = self.norm_q(query)
        kv = self.norm_kv(key_value)
        need_weights = not self.training
        attended, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=key_padding_mask,
            need_weights=need_weights, average_attn_weights=True)  # weights saved for visualization
        query = query + attended  # residual connection
        # Pre-norm feed-forward + residual (same pattern)
        query = query + self.ff(self.norm_ff(query))
        return query, attn_weights


class SelfAttentionBlock(nn.Module):
    """Pre-norm transformer self-attention block (per-modality refinement)."""

    def __init__(self, embed_dim, num_heads, dropout, attn_dropout):
        super().__init__()
        if attn_dropout is None:
            attn_dropout = dropout
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads,
                                          dropout=attn_dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        h = self.norm1(x)
        attn, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + attn
        x = x + self.ff(self.norm2(x))
        return x


class AttentionPooling(nn.Module):
    """Learned-query attention pool: a single query attends over the sequence.

    Replaces CLS-token pooling so the downstream classifier sees a learned
    weighted summary over all tokens (text padding ignored via key_padding_mask).
    """

    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, key_padding_mask=None):
        q = self.query.expand(x.size(0), -1, -1)
        out, _ = self.attn(q, x, x, key_padding_mask=key_padding_mask, need_weights = False)
        return self.norm(out.squeeze(1))


class AsymmetricCrossModalFusion(nn.Module):
    """Asymmetric cross-modal fusion stacked `depth` times -- the core research contribution.

    Each layer:
      Step 1: per-stream self-attention refines within-modality structure
      Step 2: image attends to text             -> img  (visually-grounded image features)
      Step 3: text attends to the attended img  -> txt  (NOT the raw image!)
    Each direction has independently learned weights at every layer (= 2*depth cross blocks
    plus 2*depth self-attn blocks). The cross-layer information flow lets text representations
    leverage progressively-refined image-text grounding from prior layers.
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None, depth=FUSION_DEPTH):
        super().__init__()
        self.depth = depth
        self.sa_img = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.sa_txt = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.i2t_layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.t2i_layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])

    def forward(self, image_features, text_features, text_padding_mask=None):
        img, txt = image_features, text_features
        last_i2t = last_t2i = None
        for i in range(self.depth):
            img = self.sa_img[i](img)
            txt = self.sa_txt[i](txt, key_padding_mask=text_padding_mask)
            img, last_i2t = self.i2t_layers[i](
                query=img, key_value=txt, key_padding_mask=text_padding_mask)
            txt, last_t2i = self.t2i_layers[i](query=txt, key_value=img)
        # Return last-layer attention so visualization cells stay unchanged.
        return img, txt, last_i2t, last_t2i


class SymmetricCrossModalFusion(nn.Module):
    """Symmetric cross-modal fusion stacked `depth` times (baseline).

    Per-stream self-attention has independent weights per layer (same as asymmetric).
    The only weight-sharing difference vs. asymmetric is in the cross-attention block:
    a single shared CrossAttentionBlock per layer is used for both i2t and t2i
    directions in parallel on the previous layer's outputs.
    """

    def __init__(self, embed_dim, num_heads=8, dropout=0.1, attn_dropout=None, depth=FUSION_DEPTH):
        super().__init__()
        self.depth = depth
        self.sa_img = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.sa_txt = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])
        self.shared_layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout, attn_dropout)
            for _ in range(depth)
        ])

    def forward(self, image_features, text_features, text_padding_mask=None):
        img, txt = image_features, text_features
        last_i2t = last_t2i = None
        for i in range(self.depth):
            img = self.sa_img[i](img)
            txt = self.sa_txt[i](txt, key_padding_mask=text_padding_mask)
            shared = self.shared_layers[i]
            new_img, last_i2t = shared(
                query=img, key_value=txt, key_padding_mask=text_padding_mask)
            new_txt, last_t2i = shared(query=txt, key_value=img)
            img, txt = new_img, new_txt
        return img, txt, last_i2t, last_t2i


### Full VQA Models

Pipeline: encode image + text → cross-modal fusion → mean-pool → concatenate → MLP classifier

The two models are identical except for the fusion layer.

In [15]:
class GenerativeVQAModel(nn.Module):
    """Frozen vision (ViT-bigG) + frozen text (FlanT5 encoder) + trainable
    asymmetric/symmetric cross-modal fusion + frozen FlanT5 decoder generative head.

    Forward in train mode returns the seq2seq loss. Inference is via .generate().

    The vision encoder is kept as a submodule for ad-hoc / debug use, but the
    training path consumes PRECOMPUTED vision features (see encode signature).
    """

    def __init__(self, fusion_type, t5_model, vision_encoder, text_encoder,
                 embed_dim=EMBED_DIM, num_heads=NUM_HEADS, depth=FUSION_DEPTH,
                 dropout=DROPOUT, attn_dropout=ATTN_DROPOUT,
                 vision_dim=VISION_DIM):
        super().__init__()
        assert fusion_type in ("asymmetric", "symmetric"), fusion_type
        self.fusion_type = fusion_type

        # Frozen backbones (passed in so weights are shared with the LM head)
        self.vision_encoder = vision_encoder    # frozen; not called per-step (features precomputed)
        self.text_encoder = text_encoder        # frozen (T5 encoder)
        self.t5 = t5_model                      # decoder used for generation; encoder frozen above

        # Freeze the whole T5 (encoder + decoder + LM head). Only fusion + projections train.
        for p in self.t5.parameters():
            p.requires_grad = False

        # Trainable projections into the shared fusion space
        self.vision_proj = nn.Sequential(
            nn.Linear(vision_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(0.1),
        )
        self.text_proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(0.1),
        )

        # The asymmetric/symmetric fusion stack -- the trainable centerpiece.
        if fusion_type == "asymmetric":
            self.fusion = AsymmetricCrossModalFusion(
                embed_dim, num_heads, dropout, attn_dropout, depth=depth)
        else:
            self.fusion = SymmetricCrossModalFusion(
                embed_dim, num_heads, dropout, attn_dropout, depth=depth)

        # Final LayerNorm before handing tokens to the frozen decoder.
        self.out_norm = nn.LayerNorm(embed_dim)

    def encode(self, vision_feats, q_input_ids, q_attention_mask):
        """Run frozen text encoder + trainable fusion on PRECOMPUTED vision features.

        vision_feats: (B, Nv, vision_dim) read from the precomputed-features h5.
        """
        with torch.inference_mode():
            txt_feats = self.text_encoder(q_input_ids, q_attention_mask)     # (B, L, 1024) bf16

        # Projections are in fp32; cast inputs accordingly. .float() is
        # compile-safe (avoids subscripting self.vision_proj after Module.compile()).
        # Under autocast(bf16), the matmul itself still runs in bf16.
        img_feats = self.vision_proj(vision_feats.float())
        # text_encoder runs under inference_mode -> output must be cloned before
        # entering a grad-tracking op.
        txt_feats = self.text_proj(txt_feats.float().clone())

        text_pad_mask = q_attention_mask == 0                                # True where pad
        img_out, txt_out, attn_i2t, attn_t2i = self.fusion(
            img_feats, txt_feats, text_pad_mask)

        # Concatenate fused text + fused image tokens as encoder context for the decoder.
        fused = torch.cat([txt_out, img_out], dim=1)                         # (B, L+Nv, D)
        fused = self.out_norm(fused)

        img_mask = torch.ones(
            img_out.shape[0], img_out.shape[1],
            dtype=q_attention_mask.dtype, device=q_attention_mask.device)
        fused_mask = torch.cat([q_attention_mask, img_mask], dim=1)          # (B, L+Nv)

        return fused, fused_mask, {"img_to_txt": attn_i2t, "txt_to_img": attn_t2i}

    def forward(self, vision_feats, q_input_ids, q_attention_mask, labels):
        fused, fused_mask, attn = self.encode(vision_feats, q_input_ids, q_attention_mask)
        out = self.t5(
            encoder_outputs=BaseModelOutput(last_hidden_state=fused),
            attention_mask=fused_mask,
            labels=labels,
        )
        return out.loss, out.logits, attn

    @torch.no_grad()
    def generate_answers(self, vision_feats, q_input_ids, q_attention_mask,
                         max_new_tokens=ANSWER_MAX_LEN, num_beams=1):
        fused, fused_mask, _ = self.encode(vision_feats, q_input_ids, q_attention_mask)
        return self.t5.generate(
            encoder_outputs=BaseModelOutput(last_hidden_state=fused),
            attention_mask=fused_mask,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            do_sample=False,
        )


### Precompute vision features (one-time)

The frozen `FrozenVisionEncoder` (OpenCLIP ViT-bigG-14, ~2.5B params) dominates
per-step training cost. Because the image transform is effectively deterministic
on this dataset (336x336 stored -> `Resize(224)` -> 224x224, then either
`CenterCrop(224)` or `RandomCrop(224)` is a no-op), we can precompute its output
once and stream features instead of images.

Output: `vqa_vision_feats_bigG.h5`
- `features`: (N_images, Nv, 1664) float16 -- ~70 GB
- `image_ids`: (N_images,) int64


In [16]:
# --- Precompute frozen vision encoder features to HDF5 (one-time, idempotent) ---
feats_h5_path = DATA_DIR / "vqa_vision_feats_bigG.h5"

if feats_h5_path.exists():
    print(f"Vision feature cache already exists: {feats_h5_path}")
else:
    print(f"Building vision feature cache at {feats_h5_path}")
    _precompute_transform = get_image_transform("val")  # deterministic Resize+CenterCrop+Normalize
    _precompute_encoder = FrozenVisionEncoder().to(device, dtype=torch.bfloat16)
    _precompute_encoder.eval()

    with h5py.File(h5_path, "r") as img_h5:
        all_image_ids = img_h5["image_ids"][:]
        n_images = len(all_image_ids)
        print(f"  {n_images:,} images to encode")

        # Probe output shape with a single forward.
        sample_img = Image.fromarray(img_h5["images"][0])
        sample_tensor = _precompute_transform(sample_img).unsqueeze(0).to(device, dtype=torch.bfloat16)
        with torch.inference_mode(), torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16):
            probe = _precompute_encoder(sample_tensor)
        n_tokens, n_dim = probe.shape[1], probe.shape[2]
        print(f"  feature shape per image: ({n_tokens}, {n_dim})")

        precompute_batch = 64
        with h5py.File(feats_h5_path, "w") as feats_h5:
            feats_ds = feats_h5.create_dataset(
                "features", shape=(n_images, n_tokens, n_dim),
                dtype=np.float16, chunks=(1, n_tokens, n_dim),
            )
            ids_ds = feats_h5.create_dataset(
                "image_ids", shape=(n_images,), dtype=np.int64)
            ids_ds[:] = all_image_ids

            with tqdm(total=n_images, desc="Encoding vision features", unit="image") as pbar:
                for start in range(0, n_images, precompute_batch):
                    end = min(start + precompute_batch, n_images)
                    batch_imgs = []
                    for j in range(start, end):
                        pil = Image.fromarray(img_h5["images"][j])
                        batch_imgs.append(_precompute_transform(pil))
                    batch_tensor = torch.stack(batch_imgs, dim=0).to(
                        device, dtype=torch.bfloat16, non_blocking=True)
                    with torch.inference_mode(), torch.amp.autocast(
                        device_type=device.type, dtype=torch.bfloat16
                    ):
                        feats = _precompute_encoder(batch_tensor)
                    feats_ds[start:end] = feats.to(torch.float16).cpu().numpy()
                    pbar.update(end - start)
                    pbar.set_postfix(batch=end - start)

    del _precompute_encoder
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print(f"Saved vision features to {feats_h5_path}")


Building vision feature cache at /content/data/vqa_vision_feats_bigG.h5


open_clip_model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

  123,287 images to encode
  feature shape per image: (256, 1664)


Encoding vision features:   0%|          | 0/123287 [00:00<?, ?image/s]

Saved vision features to /content/data/vqa_vision_feats_bigG.h5


### Build datasets + DataLoaders (uses precomputed feature h5)

In [17]:
# Build train/val datasets + DataLoaders against the precomputed feature h5.
train_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_train2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_train2014_annotations.json",
    vision_feats_h5_path=feats_h5_path,
    t5_tokenizer=t5_tokenizer,
    max_question_len=MAX_QUESTION_LEN,
    answer_max_len=ANSWER_MAX_LEN,
    max_samples=MAX_SAMPLES,
)
val_ds = VQADataset(
    questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
    annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
    vision_feats_h5_path=feats_h5_path,
    t5_tokenizer=t5_tokenizer,
    max_question_len=MAX_QUESTION_LEN,
    answer_max_len=ANSWER_MAX_LEN,
    max_samples=MAX_SAMPLES,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=PREFETCH_FACTOR,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=PREFETCH_FACTOR,
)

# Fixed validation subset indices for per-epoch fast eval.
rng = np.random.default_rng(SEED)
_val_subset_n = min(EVAL_SUBSET_SIZE, len(val_ds))
val_subset_indices = rng.choice(len(val_ds), size=_val_subset_n, replace=False).tolist()
val_subset_loader = DataLoader(
    torch.utils.data.Subset(val_ds, val_subset_indices),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=PREFETCH_FACTOR,
)

print(f"Train: {len(train_ds):,} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} samples ({len(val_loader)} batches)")
print(f"Val subset (per-epoch fast eval): {len(val_subset_indices):,} samples")


Train: 443,757 samples (3467 batches)
Val:   214,354 samples (1675 batches)
Val subset (per-epoch fast eval): 10,000 samples


---
## 3. Training

In [18]:
def train_one_epoch(model, loader, optimizer, scheduler, use_amp):
    """One epoch of generative training. Returns dict with train_loss."""
    model.train()
    # Backbones stay in eval mode even when model.train() is called -- BN/dropout
    # inside the frozen encoders should not be activated.
    model.vision_encoder.eval()
    model.text_encoder.eval()
    model.t5.eval()

    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    n_seen = 0

    for batch_idx, batch in enumerate(tqdm(loader, desc="  train", leave=False)):
        vision_feats, q_ids, q_mask, labels, _answers_json = batch
        vision_feats = vision_feats.to(device, non_blocking=True)
        q_ids  = q_ids.to(device, non_blocking=True)
        q_mask = q_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            loss, _logits, _attn = model(vision_feats, q_ids, q_mask, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        if scheduler is not None:
            scheduler.step()

        bs = vision_feats.size(0)
        total_loss += loss.item() * bs
        n_seen += bs

    return {"train_loss": total_loss / max(1, n_seen)}


def _vqa_soft_acc(prediction: str, annotator_answers: list) -> float:
    """Official VQA accuracy: min(1, count_matching / 3) over 10 annotators."""
    pred_norm = normalize_answer(prediction)
    matches = sum(1 for a in annotator_answers if a == pred_norm)
    return min(1.0, matches / 3.0)


@torch.no_grad()
def evaluate_generative(model, loader, tokenizer, use_amp, num_beams=1, desc="  eval"):
    """Generate answers and score against the 10 annotator answers per question."""
    model.eval()
    vqa_acc_sum = 0.0
    n_seen = 0
    sample_preds = []  # keep a few for logging

    for batch in tqdm(loader, desc=desc, leave=False):
        vision_feats, q_ids, q_mask, _labels, answers_json = batch
        vision_feats = vision_feats.to(device, non_blocking=True)
        q_ids  = q_ids.to(device, non_blocking=True)
        q_mask = q_mask.to(device, non_blocking=True)

        amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
        with amp_ctx:
            gen_ids = model.generate_answers(
                vision_feats, q_ids, q_mask, num_beams=num_beams)
        preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

        for pred, ans_json in zip(preds, answers_json):
            ann_answers = json.loads(ans_json)
            vqa_acc_sum += _vqa_soft_acc(pred, ann_answers)
            if len(sample_preds) < 5:
                sample_preds.append((pred, ann_answers[:3]))
            n_seen += 1

    return {
        "val_vqa_acc": vqa_acc_sum / max(1, n_seen) * 100,
        "sample_preds": sample_preds,
    }


In [19]:
def build_model_and_optim():
    """Build the frozen-backbone model and optimizer fresh, respecting FUSION_TYPE."""
    set_seed(SEED)
    t5 = T5ForConditionalGeneration.from_pretrained(T5_MODEL, torch_dtype=torch.bfloat16)
    vision_encoder = FrozenVisionEncoder().to(device, dtype=torch.bfloat16)
    text_encoder = FrozenTextEncoder(t5).to(device, dtype=torch.bfloat16)
    t5 = t5.to(device)

    model = GenerativeVQAModel(
        fusion_type=FUSION_TYPE,
        t5_model=t5,
        vision_encoder=vision_encoder,
        text_encoder=text_encoder,
    ).to(device)

    # Vision features are precomputed, so the vision encoder is dead weight on
    # GPU during training. Move it to CPU to free ~5 GB for batch headroom.
    # (Still callable from CPU if anyone needs it for debugging.)
    model.vision_encoder.to("cpu")
    if device.type == "cuda":
        torch.cuda.empty_cache()

    # torch.compile the trainable surface IN-PLACE via nn.Module.compile().
    # The in-place form does NOT replace self.fusion with an OptimizedModule, so
    # state_dict keys, subscription (vision_proj[0]), and resume all keep working.
    # The frozen T5 .generate() path has dynamic shapes that don't compile cleanly,
    # so it's intentionally skipped.
    try:
        model.fusion.compile(mode="reduce-overhead", dynamic=False)
        model.vision_proj.compile(mode="reduce-overhead")
        model.text_proj.compile(mode="reduce-overhead")
        print("torch.compile: in-place wrap of fusion + projections")
    except Exception as e:
        print(f"torch.compile skipped ({type(e).__name__}: {e})")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model: {FUSION_TYPE} | Trainable: {trainable:,} / {total:,} total")

    # Optimizer: separate LR for the small new projections vs the fusion stack.
    NO_DECAY_KEYS = ("bias", "LayerNorm.weight", "layer_norm.weight", "ln_")
    def _split(named):
        d, nd = [], []
        for n, p in named:
            (nd if any(k in n for k in NO_DECAY_KEYS) else d).append(p)
        return d, nd

    proj_named = [(n, p) for n, p in model.named_parameters()
                  if p.requires_grad and ("vision_proj." in n or "text_proj." in n)]
    fusion_named = [(n, p) for n, p in model.named_parameters()
                    if p.requires_grad and not ("vision_proj." in n or "text_proj." in n)]

    groups = []
    pd, pnd = _split(proj_named)
    if pd:  groups.append({"params": pd,  "lr": PROJ_LR,      "weight_decay": WEIGHT_DECAY})
    if pnd: groups.append({"params": pnd, "lr": PROJ_LR,      "weight_decay": 0.0})
    fd, fnd = _split(fusion_named)
    if fd:  groups.append({"params": fd,  "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY})
    if fnd: groups.append({"params": fnd, "lr": LEARNING_RATE, "weight_decay": 0.0})

    optimizer = torch.optim.AdamW(groups)

    import math as _math
    steps_per_epoch = max(1, len(train_loader))
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = steps_per_epoch * WARMUP_EPOCHS

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + _math.cos(_math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return model, optimizer, scheduler


# Keys whose state_dict entries we save in checkpoints. Frozen backbones (vision_encoder,
# text_encoder, t5) are huge and reloaded from HF on resume, so they're excluded.
# In-place .compile() keeps state_dict keys unchanged, so plain prefixes suffice.
TRAINABLE_PREFIXES = ("fusion.", "vision_proj.", "text_proj.", "out_norm.")


def _trainable_state_dict(model):
    return {k: v for k, v in model.state_dict().items()
            if any(k.startswith(p) for p in TRAINABLE_PREFIXES)}


def run_training(run_name=None, resume_from=None):
    """Train the generative VQA model. One run per notebook (variant = FUSION_TYPE)."""
    if run_name is None:
        run_name = f"{FUSION_TYPE}_s{SEED}"

    model, optimizer, scheduler = build_model_and_optim()
    use_amp = USE_AMP and device.type == "cuda"

    history = []
    best_val_acc = 0.0
    best_acc_epoch = None
    epochs_without_improvement = 0
    start_epoch = 1

    if resume_from is not None and Path(resume_from).exists():
        print(f"Resuming from {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        try:
            model.load_state_dict(ckpt["model_state_dict"], strict=False)
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            start_epoch = ckpt["epoch"] + 1
            history_file = METRICS_DIR / f"{run_name}_history.json"
            if history_file.exists():
                with open(history_file) as f:
                    history = json.load(f)
            if history:
                best_entry = max(history, key=lambda h: h.get("val_vqa_acc", 0))
                best_val_acc = best_entry.get("val_vqa_acc", 0)
                best_acc_epoch = best_entry["epoch"]
        except (RuntimeError, ValueError) as e:
            print(f"  Incompatible checkpoint, training fresh. ({type(e).__name__}: {str(e).splitlines()[0]})")
            start_epoch = 1; history = []; best_val_acc = 0.0; best_acc_epoch = None

    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()
        train_m = train_one_epoch(model, train_loader, optimizer, scheduler, use_amp)
        # Fast per-epoch signal on the fixed 10K val subset (generative + soft-VQA).
        val_m = evaluate_generative(model, val_subset_loader, t5_tokenizer, use_amp,
                                    desc="  val-subset")
        elapsed = time.time() - t0

        sample_preds = val_m.pop("sample_preds", [])
        epoch_data = {"epoch": epoch, **train_m, **val_m, "elapsed_s": round(elapsed, 1)}
        history.append(epoch_data)

        print(f"  Epoch {epoch}/{EPOCHS} | loss {train_m['train_loss']:.4f} | "
              f"subset vqa_acc {val_m['val_vqa_acc']:.2f}% | {elapsed:.0f}s")
        for pred, gold in sample_preds[:3]:
            print(f"    pred={pred!r:20s}  gold[:3]={gold}")

        trainable_sd = _trainable_state_dict(model)
        epoch_ckpt = CHECKPOINT_DIR / f"{run_name}_epoch{epoch}.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": trainable_sd,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "metrics": epoch_data,
            "fusion_type": FUSION_TYPE,
        }, epoch_ckpt)

        improved = val_m["val_vqa_acc"] > best_val_acc
        if improved:
            best_val_acc = val_m["val_vqa_acc"]
            best_acc_epoch = epoch
            epochs_without_improvement = 0
            torch.save({"model_state_dict": trainable_sd, **epoch_data},
                       CHECKPOINT_DIR / f"{run_name}_best.pt")
            print(f"    New best: {best_val_acc:.2f}%")
        else:
            epochs_without_improvement += 1

        # Keep only latest + best to cap disk.
        keep = {epoch_ckpt}
        if best_acc_epoch is not None:
            keep.add(CHECKPOINT_DIR / f"{run_name}_epoch{best_acc_epoch}.pt")
        for stale in CHECKPOINT_DIR.glob(f"{run_name}_epoch*.pt"):
            if stale not in keep:
                stale.unlink(missing_ok=True)

        with open(METRICS_DIR / f"{run_name}_history.json", "w") as f:
            json.dump(history, f, indent=2)

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"  Early stop: no subset-acc improvement for {EARLY_STOP_PATIENCE} epochs. "
                  f"Best={best_val_acc:.2f}% at epoch {best_acc_epoch}.")
            break

    print(f"Training complete. Best subset val_vqa_acc: {best_val_acc:.2f}% (epoch {best_acc_epoch})")

    # Final full-val eval on the best checkpoint.
    if best_acc_epoch is not None:
        best_ckpt = torch.load(CHECKPOINT_DIR / f"{run_name}_best.pt", map_location=device)
        model.load_state_dict(best_ckpt["model_state_dict"], strict=False)
    full_m = evaluate_generative(model, val_loader, t5_tokenizer, use_amp,
                                 num_beams=4, desc="  full-val")
    print(f"Full-val generative VQA acc (beam=4): {full_m['val_vqa_acc']:.2f}%")
    with open(METRICS_DIR / f"{run_name}_final.json", "w") as f:
        json.dump({"final_val_vqa_acc": full_m["val_vqa_acc"]}, f, indent=2)

    return model, history


### 3. Train the selected fusion variant (FUSION_TYPE)

This notebook trains one variant per run. To produce the comparison, run the sibling notebook (`02_train_generative_frozen_symmetric.ipynb`) which is byte-identical except for the `FUSION_TYPE` config line.

In [20]:
# (intentionally empty)
# In the generative-frozen design, this notebook trains exactly one fusion variant
# selected by FUSION_TYPE in the config cell. The single training driver lives
# in the next code cell (was the asymmetric-driver cell in the original notebook).
pass


### 3.1 Run training

In [ ]:
# Single training run for the selected FUSION_TYPE.
run_name = f"{FUSION_TYPE}_s{SEED}"
asym_ckpts = sorted(
    CHECKPOINT_DIR.glob(f"{run_name}_epoch*.pt"),
    key=lambda p: int(p.stem.rsplit("epoch", 1)[1]),
)
resume = asym_ckpts[-1] if asym_ckpts else None
if resume is not None:
    print("=" * 78)
    print(f"WARNING: about to auto-resume from {resume}")
    print("If those checkpoints are from a pre-A100 / pre-precompute regime")
    print("(different BS / LR / scheduler), delete them first to start fresh:")
    print(f"  for p in {CHECKPOINT_DIR}/*_epoch*.pt: p.unlink()")
    print("=" * 78)
trained_model, training_history = run_training(resume_from=resume)


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

  Epoch 4/18 | loss 2.5746 | subset vqa_acc 41.49% | 1780s
    pred='white'               gold[:3]=['white', 'white', 'white']
    pred='yes'                 gold[:3]=['yes', 'yes', 'yes']
    pred='white'               gold[:3]=['burnt red', 'red', 'brown']
    New best: 41.49%


  train:   0%|          | 0/3467 [00:00<?, ?it/s]

  val-subset:   0%|          | 0/79 [00:00<?, ?it/s]

  Epoch 5/18 | loss 2.4657 | subset vqa_acc 43.47% | 1784s
    pred='white'               gold[:3]=['white', 'white', 'white']
    pred='yes'                 gold[:3]=['yes', 'yes', 'yes']
    pred='white'               gold[:3]=['burnt red', 'red', 'brown']
    New best: 43.47%


  train:   0%|          | 0/3467 [00:00<?, ?it/s]

### 3.3 Load Models from .pt File

In [ ]:
# # Rebuild the model architecture (match the training-time construction)
# _common = dict(attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT)
# if FREEZE_ENCODERS:
#     symmetric_model = SymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, **_common).to(device)
# else:
#     symmetric_model = SymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
#                                             freeze_encoders=False, **_common).to(device)
#
# sym_checkpoint = torch.load(CHECKPOINT_DIR / "symmetric_s42_best.pt", map_location=device)
# symmetric_model.load_state_dict(sym_checkpoint["model_state_dict"])
# symmetric_model.eval()
#
# sym_history_file = METRICS_DIR / "symmetric_s42_history.json"
# if sym_history_file.exists():
#     with open(sym_history_file, "r") as f:
#         symmetric_history = json.load(f)
# else:
#     symmetric_history = []
#     print(f"WARNING: {sym_history_file.name} missing — training did not complete.")


In [ ]:
# === DISABLED in generative-frozen variant ===
# Cell referenced removed names: FREEZE_ENCODERS, AsymmetricVQAModelE2E, AsymmetricVQAModel(, NUM_ANSWERS
# (left commented for the user to port or delete)
# # Rebuild the model architecture (match the training-time construction in run_training)
# _common = dict(attn_dropout=ATTN_DROPOUT, cls_dropout=CLS_DROPOUT)
# if FREEZE_ENCODERS:
#     asymmetric_model = AsymmetricVQAModel(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT, **_common).to(device)
# else:
#     asymmetric_model = AsymmetricVQAModelE2E(NUM_ANSWERS, EMBED_DIM, NUM_HEADS, DROPOUT,
#                                               freeze_encoders=False, **_common).to(device)
#
# asym_checkpoint = torch.load(CHECKPOINT_DIR / "asymmetric_s42_best.pt", map_location=device)
# asymmetric_model.load_state_dict(asym_checkpoint["model_state_dict"])
# asymmetric_model.eval()
#
# # Load its history so the graphs plot correctly
# asym_history_file = METRICS_DIR / "asymmetric_s42_history.json"
# if asym_history_file.exists():
#     with open(asym_history_file, "r") as f:
#         asymmetric_history = json.load(f)
# else:
#     asymmetric_history = []
#     print(f"WARNING: {asym_history_file.name} missing — training did not complete.")


---
## 4. Evaluation

In [ ]:
# Load the most-recent histories for both fusion types if available, so the
# comparison cells below can show side-by-side metrics. Each notebook trains
# one variant and writes its own history file.
def _load_history(variant):
    fp = METRICS_DIR / f"{variant}_s{SEED}_history.json"
    if fp.exists():
        with open(fp) as f:
            return json.load(f)
    return None

asymmetric_history = _load_history("asymmetric")
symmetric_history  = _load_history("symmetric")

print(f"asymmetric history: {'loaded' if asymmetric_history else 'not found'}")
print(f"symmetric history:  {'loaded' if symmetric_history else 'not found'}")


---
## 5. Visualization

### 5.1 Training Curves

In [ ]:
histories = {}
if asymmetric_history is not None:
    histories["Asymmetric"] = asymmetric_history
if symmetric_history is not None:
    histories["Symmetric"] = symmetric_history

if histories:
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, h in histories.items():
        epochs = [e["epoch"] for e in h]
        accs = [e["val_vqa_acc"] for e in h]
        ax.plot(epochs, accs, marker="o", label=name)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Val subset VQA accuracy (%)")
    ax.set_title("Validation accuracy per epoch (subset)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "val_accuracy_per_epoch.png", dpi=150)
    plt.show()
else:
    print("No histories yet -- run training first.")


### 5.2 Comparison Bar Chart

In [ ]:
metric_names = ["val_vqa_acc"]
if histories:
    final = {n: h[-1] for n, h in histories.items()}
    fig, ax = plt.subplots(figsize=(6, 4))
    xs = np.arange(len(metric_names))
    width = 0.35
    for i, (name, m) in enumerate(final.items()):
        vals = [m.get(k, 0) for k in metric_names]
        ax.bar(xs + i * width, vals, width, label=name)
    ax.set_xticks(xs + width / 2)
    ax.set_xticklabels(metric_names)
    ax.set_ylabel("Score")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "final_metric_bars.png", dpi=150)
    plt.show()
else:
    print("No histories yet -- run training first.")


### Visualization PreparationRe-attach frozen encoders to trained fusion models and reload the raw datasetfor visualization. Training used precomputed features; visualization needs rawimages and question strings.

In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# # Re-instantiate the raw dataset for visualization (images + question strings)
# raw_val_ds = VQADataset(
#     questions_file=DATA_DIR / "questions" / "v2_OpenEnded_mscoco_val2014_questions.json",
#     annotations_file=DATA_DIR / "answers" / "v2_mscoco_val2014_annotations.json",
#     h5_path=h5_path,
#     answer_to_idx=answer_to_idx,
#     max_question_len=MAX_QUESTION_LEN,
#     transform=get_image_transform("val"),
#     max_samples=MAX_SAMPLES,
# )
#
# raw_val_loader = DataLoader(raw_val_ds, batch_size=BATCH_SIZE, shuffle=False,
#                             num_workers=NUM_WORKERS, pin_memory=True)
#
# if FREEZE_ENCODERS:
#     class EndToEndVQAWrapper(nn.Module):
#         """Wraps a trained fusion model with fresh frozen encoders for visualization."""
#
#         def __init__(self, trained_fusion_model, embed_dim=EMBED_DIM):
#             super().__init__()
#             self.image_encoder = ImageEncoder(embed_dim, freeze=True)
#             self.text_encoder = TextEncoder(embed_dim, freeze=True)
#             self.trained_model = trained_fusion_model
#
#         def forward(self, images, input_ids, attention_mask):
#             img_feats = self.image_encoder(images)
#             txt_feats = self.text_encoder(input_ids, attention_mask)
#             return self.trained_model(img_feats, txt_feats, attention_mask)
#
#     set_seed(SEED)
#     e2e_asymmetric = EndToEndVQAWrapper(asymmetric_model).to(device)
#     # e2e_symmetric = EndToEndVQAWrapper(symmetric_model).to(device)
#     # e2e_models_dict = {"Asymmetric": e2e_asymmetric, "Symmetric": e2e_symmetric}
#     e2e_models_dict = {"Asymmetric": e2e_asymmetric}
# else:
#     # Models are already end-to-end — use them directly
#     # e2e_models_dict = {"Asymmetric": asymmetric_model, "Symmetric": symmetric_model}
#     e2e_models_dict = {"Asymmetric": asymmetric_model}
#
# print(f"raw_val_ds: {len(raw_val_ds):,} samples")
# print("End-to-end models ready for visualization.")


### 5.3 Attention Heatmaps

Visualize what the asymmetric model attends to: which image regions light up for each question word, and which words are most important for each image patch.

In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# # Visualization helpers
# MEAN = np.array([0.485, 0.456, 0.406])
# STD  = np.array([0.229, 0.224, 0.225])
#
#
# def denormalize(img_tensor):
#     """Convert normalised (3,H,W) tensor to (H,W,3) uint8 array."""
#     img = img_tensor.cpu().numpy().transpose(1, 2, 0)
#     img = img * STD + MEAN
#     return np.clip(img * 255, 0, 255).astype(np.uint8)
#
#
# def decode_tokens(input_ids):
#     """Decode token IDs to readable strings."""
#     tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
#     return [tokenizer.decode(tid) for tid in input_ids.tolist()]
#
#
# @torch.no_grad()
# def get_attention_weights(model, image, input_ids, attention_mask):
#     """Run a single sample and return attention weight dicts."""
#     model.eval()
#     img = image.unsqueeze(0).to(device)
#     ids = input_ids.unsqueeze(0).to(device)
#     mask = attention_mask.unsqueeze(0).to(device)
#     _, attn = model(img, ids, mask)
#     return {k: v.cpu() for k, v in attn.items()}
#
#
# def plot_image_attention(attn_t2i, image_tensor, tokens, question, top_tokens=4):
#     """Overlay text->image attention heatmaps on the original image."""
#     img_np = denormalize(image_tensor)
#     attn = attn_t2i.squeeze(0).numpy()  # (N_txt, N_img)
#
#     grid_size = int(np.sqrt(attn.shape[1] - 1))  # 24 for DINOv2 ViT-g/14 at 336x336 (14-px patches)
#     attn_spatial = attn[:, 1:]  # drop CLS column
#
#     # Count how many actual words exist (ignoring padding)
#     real_token_count = len([t for t in tokens if t not in ['<pad>']])
#
#     # Only average the attention maps of the real tokens
#     combined = attn_spatial[:real_token_count].mean(axis=0).reshape(grid_size, grid_size)
#
#     combined_resized = np.array(
#         Image.fromarray(combined).resize(img_np.shape[:2][::-1], Image.BILINEAR))
#
#     token_importance = attn_spatial.sum(axis=1)
#     top_idx = token_importance.argsort()[-top_tokens:][::-1]
#
#     n_cols = min(top_tokens, len(top_idx)) + 1
#     fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
#     if n_cols == 1:
#         axes = [axes]
#
#     axes[0].imshow(img_np)
#     axes[0].imshow(combined_resized, alpha=0.5, cmap="jet")
#     axes[0].set_title("Combined")
#     axes[0].axis("off")
#
#     for i, idx in enumerate(top_idx):
#         if i + 1 >= len(axes):
#             break
#         token_attn = attn_spatial[idx].reshape(grid_size, grid_size)
#         token_resized = np.array(
#             Image.fromarray(token_attn).resize(img_np.shape[:2][::-1], Image.BILINEAR))
#         label = tokens[idx] if tokens else f"token {idx}"
#         axes[i + 1].imshow(img_np)
#         axes[i + 1].imshow(token_resized, alpha=0.5, cmap="jet")
#         axes[i + 1].set_title(f'"{label}"')
#         axes[i + 1].axis("off")
#
#     fig.suptitle(question, fontsize=12)
#     fig.tight_layout()
#     return fig
#
#
# def plot_text_attention(attn_img_to_txt, tokens, question):
#     """Bar chart of image->text attention per token."""
#     attn = attn_img_to_txt.squeeze(0).numpy()
#     token_weights = attn.mean(axis=0)
#
#     fig, ax = plt.subplots(figsize=(6, max(3, len(tokens) * 0.35)))
#     y_pos = np.arange(len(tokens))
#     ax.barh(y_pos, token_weights, color="steelblue")
#     ax.set_yticks(y_pos)
#     ax.set_yticklabels(tokens, fontsize=9)
#     ax.invert_yaxis()
#     ax.set_xlabel("Mean attention weight")
#     ax.set_title(f"Image -> Text attention\n{question}")
#     fig.tight_layout()
#     return fig


In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# # Generate attention maps for sample validation images
# for i in range(min(5, len(raw_val_ds))):
#     image, input_ids, attention_mask, answer_target = raw_val_ds[i]
#     question = raw_val_ds.samples[i]["question"]
#     tokens = decode_tokens(input_ids)
#
#     attn = get_attention_weights(e2e_models_dict["Asymmetric"], image, input_ids, attention_mask)
#
#     fig = plot_image_attention(attn["txt_to_img"], image, tokens, question)
#     fig.savefig(FIGURES_DIR / f"attn_img_{i}.png", dpi=150, bbox_inches="tight")
#     plt.show()
#
#     fig = plot_text_attention(attn["img_to_txt"], tokens, question)
#     fig.savefig(FIGURES_DIR / f"attn_txt_{i}.png", dpi=150, bbox_inches="tight")
#     plt.show()


### 5.4 Qualitative Comparison Grid

Side-by-side predictions and attention maps for both models on the same samples.

In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# @torch.no_grad()
# def qualitative_grid(models, dataset, idx_to_answer, n_samples=6, save_path=None):
#     """Grid of qualitative examples comparing models."""
#     sample_indices = torch.randperm(len(dataset))[:n_samples].tolist()
#     model_names = list(models.keys())
#     n_cols = 1 + len(model_names)
#     n_rows = len(sample_indices)
#
#     fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
#     if n_rows == 1:
#         axes = axes[np.newaxis, :]
#
#     for row, idx in enumerate(sample_indices):
#         image, input_ids, attention_mask, answer_target = dataset[idx]
#         question = dataset.samples[idx]["question"]
#         gt_answer = idx_to_answer[answer_target.argmax().item()]
#         img_np = denormalize(image)
#
#         # Column 0: original image + question
#         axes[row, 0].imshow(img_np)
#         axes[row, 0].set_title(f"Q: {question}\nGT: {gt_answer}", fontsize=9)
#         axes[row, 0].axis("off")
#
#         # Remaining columns: one per model
#         for col, name in enumerate(model_names, start=1):
#             model = models[name]
#             attn = get_attention_weights(model, image, input_ids, attention_mask)
#
#             img_t = image.unsqueeze(0).to(device)
#             ids_t = input_ids.unsqueeze(0).to(device)
#             mask_t = attention_mask.unsqueeze(0).to(device)
#             logits, _ = model(img_t, ids_t, mask_t)
#             pred_answer = idx_to_answer.get(logits.argmax(dim=1).item(), "???")
#
#             # Attention heatmap
#             attn_t2i = attn["txt_to_img"].squeeze(0).numpy()
#             grid_size = int(np.sqrt(attn_t2i.shape[1] - 1))
#             combined = attn_t2i[:, 1:].mean(axis=0).reshape(grid_size, grid_size)
#             combined_resized = np.array(
#                 Image.fromarray(combined).resize(img_np.shape[:2][::-1], Image.BILINEAR))
#
#             axes[row, col].imshow(img_np)
#             axes[row, col].imshow(combined_resized, alpha=0.5, cmap="jet")
#             marker = "correct" if pred_answer == gt_answer else "wrong"
#             axes[row, col].set_title(f"{name}\nPred: {pred_answer} ({marker})", fontsize=9)
#             axes[row, col].axis("off")
#
#     fig.tight_layout()
#     if save_path:
#         fig.savefig(save_path, dpi=150, bbox_inches="tight")
#     return fig


In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# fig = qualitative_grid(
#     e2e_models_dict, raw_val_ds, idx_to_answer, n_samples=6,
#     save_path=str(FIGURES_DIR / "qualitative_grid.png"))
# plt.show()


In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# # @torch.no_grad()
# # def error_analysis(models, loader, idx_to_answer, skip_n=1):
# #     print(f"Running Error Analysis... skipping {skip_n} cases")
# #     asym_model = models["Asymmetric"]
# #     sym_model = models["Symmetric"]
# #
# #     asym_model.eval()
# #     sym_model.eval()
# #
# #     cases_found = 0
# #
# #     for images, input_ids, attention_mask, answers in loader:
# #         images = images.to(device)
# #         input_ids = input_ids.to(device)
# #         attention_mask = attention_mask.to(device)
# #         targets = answers.to(device).argmax(dim=1)
# #
# #         asym_logits, _ = asym_model(images, input_ids, attention_mask)
# #         sym_logits, _ = sym_model(images, input_ids, attention_mask)
# #
# #         asym_preds = asym_logits.argmax(dim=1)
# #         sym_preds = sym_logits.argmax(dim=1)
# #
# #         # Find condition: Asymmetric correct AND Symmetric wrong
# #         mask = (asym_preds == targets) & (sym_preds != targets)
# #         divergent_indices = mask.nonzero(as_tuple=True)[0]
# #
# #         for idx in divergent_indices:
# #             if cases_found < skip_n:
# #                 cases_found += 1
# #                 continue
# #
# #             print("--- Found a divergent case ---")
# #             print(f"Target Answer: {idx_to_answer[targets[idx].item()]}")
# #             print(f"Asymmetric Guess: {idx_to_answer[asym_preds[idx].item()]} (Correct)")
# #             print(f"Symmetric Guess: {idx_to_answer[sym_preds[idx].item()]} (Wrong)")
# #
# #             # Extract the specific sample for visualization
# #             img = images[idx].cpu()
# #             ids = input_ids[idx].cpu()
# #             mask_ = attention_mask[idx].cpu()
# #
# #             # Decode tokens to reconstruct a readable question
# #             tokens = decode_tokens(ids)
# #             clean_tokens = [t.replace('\u0120', '') for t in tokens if t not in ['<pad>', '<s>', '</s>']]
# #             question_str = " ".join(clean_tokens).strip() + "?"
# #             print(f"Question: {question_str}")
# #
# #             # Get attention weights for BOTH models
# #             attn_asym = get_attention_weights(asym_model, img, ids, mask_)
# #             attn_sym = get_attention_weights(sym_model, img, ids, mask_)
# #
# #             print("\n=== ASYMMETRIC MODEL ATTENTION (Correct Guess) ===")
# #             fig_img_asym = plot_image_attention(attn_asym["txt_to_img"], img, tokens, question_str)
# #             plt.show()
# #             fig_txt_asym = plot_text_attention(attn_asym["img_to_txt"], tokens, question_str)
# #             plt.show()
# #
# #             print("\n=== SYMMETRIC MODEL ATTENTION (Wrong Guess) ===")
# #             fig_img_sym = plot_image_attention(attn_sym["txt_to_img"], img, tokens, question_str)
# #             plt.show()
# #             fig_txt_sym = plot_text_attention(attn_sym["img_to_txt"], tokens, question_str)
# #             plt.show()
# #
# #             return
# #
# # error_analysis(e2e_models_dict, raw_val_loader, idx_to_answer, skip_n=10)


# Task
Categorize the validation questions by type (e.g., 'Is/Are', 'How many', 'What color') and compute the accuracy for both the symmetric and asymmetric models per category, generating a grouped bar chart to visualize the results. Additionally, conduct a comprehensive modality ablation test by evaluating both models under three conditions: Full Data, Image-Blind (zeroed images), and Text-Blind (zeroed input IDs/masks), and generate a grouped bar chart showing the degradation. Finally, provide a brief summary of the insights gained from these new evaluation metrics to help frame the presentation.

## Question Type Accuracy Breakdown

### Subtask:
Categorize validation questions by type, calculate accuracy per category for both models, and plot a grouped bar chart.


In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# from collections import defaultdict
#
# # 1. Categorize question function
# def categorize_question(question):
#     q_lower = question.lower().strip()
#     if q_lower.startswith(('is ', 'are ', 'was ', 'were ', 'does ', 'do ', 'has ', 'have ', 'can ', 'could ', 'would ', 'should ')):
#         return 'Yes/No'
#     elif q_lower.startswith('how many'):
#         return 'Count'
#     elif q_lower.startswith('what color'):
#         return 'Color'
#     else:
#         return 'Other'
#
# @torch.no_grad()
# def evaluate_by_question_type(models, loader, dataset):
#     """Evaluate VQA accuracy broken down by question type.
#     Uses loader for model inference, raw dataset for question text."""
#     for model in models.values():
#         model.eval()
#
#     category_scores = {name: defaultdict(list) for name in models.keys()}
#     global_idx = 0
#
#     use_amp = USE_AMP and device.type == "cuda"
#
#     for inp1, inp2, masks, answers in tqdm(loader, desc="Evaluating by question type"):
#         inp1    = inp1.to(device)
#         inp2    = inp2.to(device)
#         masks   = masks.to(device)
#         answers = answers.to(device)
#
#         batch_size = inp1.size(0)
#
#         preds_dict = {}
#         for name, model in models.items():
#             amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
#             with amp_ctx:
#                 logits, _ = model(inp1, inp2, masks)
#             preds_dict[name] = logits.argmax(dim=1)
#
#         # 3. Calculate category score per sample
#         for i in range(batch_size):
#             question = dataset.samples[global_idx + i]["question"]
#             category = categorize_question(question)
#
#             for name, preds in preds_dict.items():
#                 pred_idx = preds[i]
#                 pred_soft = answers[i, pred_idx]
#                 vqa_score = torch.clamp(pred_soft * 10.0 / 3.0, max=1.0).item()
#                 category_scores[name][category].append(vqa_score)
#
#         global_idx += batch_size
#
#     # 4. Aggregate scores
#     category_acc = {name: {} for name in models.keys()}
#     for name, cat_scores in category_scores.items():
#         for cat, scores in cat_scores.items():
#             category_acc[name][cat] = np.mean(scores) * 100
#
#     return category_acc
#
# # Use precomputed models + val_loader for speed, raw_val_ds for question text
# category_acc = evaluate_by_question_type(
#     # {"Symmetric": symmetric_model, "Asymmetric": asymmetric_model},
#     {"Asymmetric": asymmetric_model},
#     val_loader, raw_val_ds)
#
# # 5. Generate a grouped bar chart
# categories = ['Yes/No', 'Count', 'Color', 'Other']
# # sym_acc = [category_acc["Symmetric"].get(cat, 0) for cat in categories]
# asym_acc = [category_acc["Asymmetric"].get(cat, 0) for cat in categories]
#
# x = np.arange(len(categories))
# width = 0.35
#
# fig, ax = plt.subplots(figsize=(8, 6))
# # bars1 = ax.bar(x - width/2, sym_acc, width, label='Symmetric', color='lightblue')
# bars2 = ax.bar(x + width/2, asym_acc, width, label='Asymmetric', color='steelblue')
#
# # for bars in [bars1, bars2]:
# for bars in [bars2]:
#     for bar in bars:
#         ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
#                 f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
#
# ax.set_ylabel('Accuracy (%)')
# ax.set_title('VQA Accuracy by Question Type')
# ax.set_xticks(x)
# ax.set_xticklabels(categories)
# ax.legend()
#
# fig.tight_layout()
# fig.savefig(FIGURES_DIR / "question_type_accuracy.png", dpi=150, bbox_inches="tight")
# plt.show()


## Modality Ablation Test

Evaluate both models under three conditions: Full Data, Image-Blind (zeroed images), and Text-Blind (zeroed input IDs/masks), and generate a grouped bar chart showing the degradation.

In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# @torch.no_grad()
# def evaluate_ablation(model, loader, condition):
#     """Evaluate model under ablation conditions."""
#     model.eval()
#     vqa_acc_sum = 0.0
#     total = 0
#     use_amp = USE_AMP and device.type == "cuda"
#
#     for inp1, inp2, masks, answers in tqdm(loader, desc=f"Eval {condition}"):
#         if condition == "Image-Blind":
#             inp1 = torch.zeros_like(inp1)
#         elif condition == "Text-Blind":
#             if FREEZE_ENCODERS:
#                 inp2 = torch.zeros_like(inp2)     # zero out precomputed text features
#             else:
#                 inp2 = torch.ones_like(inp2)      # RoBERTa padding token ID = 1
#             masks = torch.zeros_like(masks)       # zero attention mask in both cases
#
#         inp1    = inp1.to(device)
#         inp2    = inp2.to(device)
#         masks   = masks.to(device)
#         answers = answers.to(device)
#
#         amp_ctx = torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE) if use_amp else nullcontext()
#         with amp_ctx:
#             logits, _ = model(inp1, inp2, masks)
#
#         preds = logits.argmax(dim=1)
#         pred_soft = answers[torch.arange(answers.size(0), device=answers.device), preds]
#         vqa_scores = torch.clamp(pred_soft * 10.0 / 3.0, max=1.0)
#         vqa_acc_sum += vqa_scores.sum().item()
#         total += answers.size(0)
#
#     return vqa_acc_sum / total * 100
#
# # Use models + val_loader
# # models_dict = {"Symmetric": symmetric_model, "Asymmetric": asymmetric_model}
# models_dict = {"Asymmetric": asymmetric_model}
#
# conditions = ["Full Data", "Image-Blind", "Text-Blind"]
# # ablation_results = {"Symmetric": {}, "Asymmetric": {}}
# ablation_results = {"Asymmetric": {}}
#
# for model_name, model in models_dict.items():
#     print(f"\nRunning ablation for {model_name}...")
#     for cond in conditions:
#         acc = evaluate_ablation(model, val_loader, cond)
#         ablation_results[model_name][cond] = acc
#         print(f"  {cond}: {acc:.2f}%")
#
# # Plotting the results
# x = np.arange(len(conditions))
# width = 0.35
#
# fig, ax = plt.subplots(figsize=(8, 6))
# # sym_accs = [ablation_results["Symmetric"][cond] for cond in conditions]
# asym_accs = [ablation_results["Asymmetric"][cond] for cond in conditions]
#
# # bars1 = ax.bar(x - width/2, sym_accs, width, label='Symmetric', color='lightblue')
# bars2 = ax.bar(x + width/2, asym_accs, width, label='Asymmetric', color='steelblue')
#
# # for bars in [bars1, bars2]:
# for bars in [bars2]:
#     for bar in bars:
#         ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
#                 f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
#
# ax.set_ylabel('Accuracy (%)')
# ax.set_title('Modality Ablation Test - Performance Degradation')
# ax.set_xticks(x)
# ax.set_xticklabels(conditions)
# ax.legend()
#
# fig.tight_layout()
# fig.savefig(FIGURES_DIR / "ablation_test.png", dpi=150, bbox_inches="tight")
# plt.show()


# Task
Create a helper function `visualize_category_example` that searches the validation dataset for a question of a specified category and plots the attention heatmaps for both models. Use this function to find and visualize examples for the 'Yes/No', 'Count', 'Color', and 'Other' categories, adding appropriate text and code cells for each, and conclude with a brief summary of these visualizations.

## Define Category Visualization Helper

### Subtask:
Create a helper function to search for and visualize a specific question category.


In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# def visualize_category_example(category, dataset, models_dict, idx_to_answer):
#     print(f"Searching for an example in category: {category}...")
#     for i in range(len(dataset)):
#         question = dataset.samples[i]["question"]
#         if categorize_question(question) == category:
#             image, input_ids, attention_mask, answer_target = dataset[i]
#             gt_answer = idx_to_answer[answer_target.argmax().item()]
#
#             img_t = image.unsqueeze(0).to(device)
#             ids_t = input_ids.unsqueeze(0).to(device)
#             mask_t = attention_mask.unsqueeze(0).to(device)
#
#             # Decode tokens to reconstruct a readable question
#             tokens = decode_tokens(input_ids)
#             clean_tokens = [t.replace('Ġ', '') for t in tokens if t not in ['<pad>', '<s>', '</s>']]
#             question_str = " ".join(clean_tokens).strip() + "?"
#
#             print("\n" + "="*40)
#             print(f"Category: {category}")
#             print(f"Question: {question_str}")
#             print(f"Target Answer: {gt_answer}")
#             print("="*40)
#
#             for model_name, model in models_dict.items():
#                 model.eval()
#                 with torch.no_grad():
#                     logits, _ = model(img_t, ids_t, mask_t)
#                 pred_answer = idx_to_answer.get(logits.argmax(dim=1).item(), "???")
#                 print(f"{model_name} Guess: {pred_answer}")
#
#             for model_name, model in models_dict.items():
#                 print(f"\n=== {model_name.upper()} MODEL ATTENTION ===")
#                 attn = get_attention_weights(model, image, input_ids, attention_mask)
#
#                 # Plot text -> image attention
#                 fig_img = plot_image_attention(attn["txt_to_img"], image, tokens, question_str)
#                 plt.show()
#
#                 # Plot image -> text attention
#                 fig_txt = plot_text_attention(attn["img_to_txt"], tokens, question_str)
#                 plt.show()
#
#             return
#
#     print(f"No example found for category: {category}")


## Visualize Yes/No Question

### Subtask:
Find and visualize an example for the 'Yes/No' question category.


In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# visualize_category_example("Yes/No", raw_val_ds, e2e_models_dict, idx_to_answer)


## Visualize Count Question

### Subtask:
Find and visualize an example for the 'Count' question category.

In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# visualize_category_example("Count", raw_val_ds, e2e_models_dict, idx_to_answer)


## Visualize Color Question

### Subtask:
Find and visualize an example for the 'Color' question category.

In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# visualize_category_example("Color", raw_val_ds, e2e_models_dict, idx_to_answer)


## Visualize Other Question

### Subtask:
Find and visualize an example for the 'Other' question category.

In [ ]:
# === VIZ CELL (disabled in generative-frozen variant) ===
# This cell references the old classification-head / e2e attention map
# helpers and is left commented for the user to port if needed.
# visualize_category_example("Other", raw_val_ds, e2e_models_dict, idx_to_answer)


## Summary of Category Visualizations

Based on the attention heatmaps across different question categories ('Yes/No', 'Count', 'Color', 'Other'), we can observe the following:

1. **Asymmetric Model Focus**: The asymmetric model often demonstrates a more focused and interpretable attention mechanism. When asked about a specific object or its attribute (like 'color' or 'count'), the text-to-image attention effectively isolates the relevant regions of the image corresponding to the target words.
2. **Symmetric Model Limitations**: The symmetric model's attention maps tend to be more diffuse. Because it uses a single shared block for both directions, it struggles to decouple the distinct tasks of 'understanding the question' and 'locating the visual evidence'.
3. **Question-Specific Grounding**: In 'Count' and 'Color' questions, grounded visual evidence is crucial. The asymmetric model's ability to first process the text and then use it as a query to attend to the image allows it to better pinpoint the items to be counted or analyzed for color, leading to more accurate predictions.

In [ ]:
!mkdir -p /content/data/zip/
!mkdir -p /content/data/images/
!cp /content/drive/MyDrive/test2015.zip /content/data/zip/

In [ ]:
!unzip -q /content/data/zip/test2015.zip -d /content/data/images/

---
## 6. Test Set Predictions Export

Run inference on the VQA v2.0 **test2015** split (no annotations) and write predictions to JSON in the official VQA submission format: `[{"question_id": int, "answer": str}, ...]`.

Test images aren't in the precomputed feature HDF5, so we load raw JPEGs from `images/test2015/` and run them through the end-to-end models (`e2e_models_dict`, which works for both frozen and unfrozen runs).

In [ ]:
# === DISABLED in generative-frozen variant ===
# Cell referenced removed names: idx_to_answer, e2e_models_dict
# (left commented for the user to port or delete)
# class VQATestDataset(Dataset):
#     """VQA test split: questions + raw JPEGs from images/test2015/, no annotations."""
#
#     def __init__(self, questions_file, images_dir, max_question_len=20,
#                  transform=None, max_samples=None):
#         self.images_dir = Path(images_dir) / "test2015"
#         self.max_question_len = max_question_len
#         self.transform = transform or get_image_transform("val")
#         self.tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
#
#         with open(questions_file) as f:
#             self.samples = json.load(f)["questions"]
#         if max_samples is not None:
#             self.samples = self.samples[:max_samples]
#
#     def __len__(self):
#         return len(self.samples)
#
#     def __getitem__(self, idx):
#         sample = self.samples[idx]
#         # test-dev2015 reuses the test2015 image pool, so the filename prefix is
#         # always COCO_test2015_* regardless of which question split we loaded.
#         img_path = self.images_dir / f"COCO_test2015_{sample['image_id']:012d}.jpg"
#         image = Image.open(img_path).convert("RGB")
#         image = self.transform(image)
#
#         encoding = self.tokenizer(
#             sample["question"],
#             max_length=self.max_question_len,
#             padding="max_length",
#             truncation=True,
#             return_tensors="pt",
#         )
#         return (
#             image,
#             encoding["input_ids"].squeeze(0),
#             encoding["attention_mask"].squeeze(0),
#             sample["question_id"],
#         )
#
#
# # Use the full test2015 split (~447K questions) — required for any VQA v2
# # eval-server submission (test-dev or test-standard). For a quick local
# # sanity check during iteration, swap to v2_OpenEnded_mscoco_test-dev2015_questions.json (~107K).
# TEST_QUESTIONS_FILE = DATA_DIR / "questions" / "v2_OpenEnded_mscoco_test2015_questions.json"
#
# test_ds = VQATestDataset(
#     questions_file=TEST_QUESTIONS_FILE,
#     images_dir=DATA_DIR / "images",
#     max_question_len=MAX_QUESTION_LEN,
#     transform=get_image_transform("val"),
#     max_samples=None,  # always export the full test split, regardless of dev MAX_SAMPLES
# )
# test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
#                          num_workers=NUM_WORKERS, pin_memory=True)
# print(f"Test: {len(test_ds):,} samples ({len(test_loader)} batches)")
#
#
# @torch.no_grad()
# def predict_test(model, loader, idx_to_answer):
#     """Return [{question_id, answer}, ...] in official VQA submission format."""
#     model.eval()
#     use_amp = USE_AMP and device.type == "cuda"
#     predictions = []
#
#     for images, input_ids, attention_mask, question_ids in tqdm(loader, desc="predict"):
#         images = images.to(device)
#         input_ids = input_ids.to(device)
#         attention_mask = attention_mask.to(device)
#
#         amp_ctx = torch.amp.autocast(device_type=device.type, dtype=AMP_DTYPE) if use_amp else nullcontext()
#         with amp_ctx:
#             logits, _ = model(images, input_ids, attention_mask)
#
#         preds = logits.argmax(dim=1).cpu().tolist()
#         for qid, p in zip(question_ids.tolist(), preds):
#             predictions.append({"question_id": int(qid), "answer": idx_to_answer[int(p)]})
#
#     return predictions
#
#
# PREDICTIONS_DIR = CHECKPOINT_DIR.parent / "predictions"
# PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
#
# # for name, model in [("asymmetric", e2e_models_dict["Asymmetric"]),
# #                     ("symmetric",  e2e_models_dict["Symmetric"])]:
# for name, model in [("asymmetric", e2e_models_dict["Asymmetric"])]:
#     preds = predict_test(model, test_loader, idx_to_answer)
#     out_path = PREDICTIONS_DIR / f"{name}_test_predictions.json"
#     with open(out_path, "w") as f:
#         json.dump(preds, f)
#     print(f"Saved {len(preds):,} {name} predictions -> {out_path}")


In [ ]:
!cp -r /content/results/* /content/drive/MyDrive/updated_unfrozen_results/